# Level 1 — Lesson 3: Build a Segmentation Model with an Open LLM

## From safe prompting to a working AI pipeline

**Teaching time:** ~45 minutes  
**Core question:** *Can I use an LLM to help me build a small AI application while keeping the code simple, testable, and understandable?*

In Lesson 1 we learned how to work incrementally with an LLM.

In Lesson 2 we asked whether the information we want to share is appropriate for the AI system we are using.

Now we combine those ideas.

We will use:

- **public** Medical Segmentation Decathlon data;
- Python;
- PyTorch;
- MONAI;
- an **LLM** of your choice as a coding collaborator.

The goal is to build:

```text
public 3D CT data
        ↓
preprocessing
        ↓
training patches
        ↓
small 3D segmentation model
        ↓
training
        ↓
whole-volume prediction
        ↓
visual inspection
```

You do not need to understanding every neural-network component. However, you should be able to judge whether LLM generated code is not only working but actually scientifically and computationally sound.  

> **Can you guide an LLM to construct a small working program, one verified step at a time?**

# 0. Which LLM should I use?

Use any capable model that you can access in an approved way. If you run this notebook under Google colab, you could, for example, directly use Google's Gemini which you can access pressing the "toggle Gemini" button at the buttom of this page.

However, the exact model is intentionally not part of the lesson because available models and interfaces change quickly.

Examples of possible access patterns include:

```text
open-weight model in a browser interface
open-weight model hosted by your institution
local model through Ollama / LM Studio
approved research compute
```

For this lesson, the model only sees:

```text
public dataset names
generic Python code
generic error messages
```

No patient or institutional data are to be entered.


# 1. Start with the specification before you ask for code

Our task is:

> Train a very small 3D model to segment the spleen from CT using a few cases from MSD Task09 Spleen.

A useful first prompt is:

> I want to build a minimal educational 3D spleen-segmentation pipeline in Google Colab using PyTorch and MONAI.  
>
> Use the public Medical Segmentation Decathlon Task09 Spleen dataset.  
>
> Constraints:
> - keep the implementation small and readable;
> - use only a few training cases so that it can run on free Colab;
> - use MONAI components where they already exist;
> - use patch-based training;
> - use a small 3D U-Net;
> - use Dice loss;
> - use sliding-window inference for validation;
> - show one final CT slice with reference and predicted contours;
> - do not introduce configuration classes, custom logging frameworks, command-line interfaces, or other infrastructure.
>
> Before writing code, propose the smallest reasonable implementation plan.  
> Identify what should be verified after each step.

### Exercise

Send this to your LLM, but do **not** ask for code yet. Check the output carefully. Is the plan sound?  

## What should a reasonable plan contain?

Something close to:

```text
1. Install/import packages.
2. Download Task09 Spleen.
3. Define training and validation transforms.
4. Create a very small train/validation subset.
5. Create DataLoaders.
6. Define a small 3D U-Net.
7. Define Dice loss and optimizer.
8. Train for a few epochs.
9. Run whole-volume sliding-window inference.
10. Visualize one prediction.
```

Now inspect what your LLM proposed.

### Check the scope

Do you think the proposed scope makes sense? Is there something you would like to add/remove?

Look specifically for scope creep. Some infrastructure may be intended for future integration in a larger project or it may be too detailed for the current task.

If needed, reply:

> **Simplify this. I want the smallest program that demonstrates the complete workflow. Remove infrastructure that is not required for this notebook.**

# 2. Ask for the data pipeline first

It is always better to ask for small chunks of code that can be verified and checked rather than directly the full repository with several subfiles and hundreds to thousands of lines of code.

For our task ask:

> Implement only the dataset download, transforms, small train/validation subsets, and DataLoaders.  
>
> Do not define a model or training loop yet.  
>
> After the code, show me exactly what tensor shapes I should print to verify that the data pipeline works.

Run the generated code. You can add it to this notebook as a new code cell.

Compare its behaviour with the reference implementation below if needed. Note, run only one of the two, either the LLM or your own code. Running both pieces of code may interfere with each other.

In [ ]:
!pip -q install "monai==1.6.1" nibabel

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

import monai
from monai.apps import DecathlonDataset
from monai.data import CacheDataset, Dataset, DataLoader
from monai.inferers import sliding_window_inference
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    RandCropByPosNegLabeld,
    RandFlipd,
    EnsureTyped,
)
from monai.utils import set_determinism

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_determinism(seed=SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("MONAI:", monai.__version__)
print("PyTorch:", torch.__version__)
print("Device:", device)

## Download the public dataset

The dataset is larger than we need, but we will train on only a few cases.

The first download may take a minute or two.

In [ ]:
DATA_DIR = Path("/content/msd")

raw_train = DecathlonDataset(
    root_dir=DATA_DIR,
    task="Task09_Spleen",
    section="training",
    transform=None,
    download=True,
    seed=SEED,
    val_frac=0.2,
    cache_rate=0.0,
)

raw_val = DecathlonDataset(
    root_dir=DATA_DIR,
    task="Task09_Spleen",
    section="validation",
    transform=None,
    download=False,
    seed=SEED,
    val_frac=0.2,
    cache_rate=0.0,
)

train_files = list(raw_train.data)[:4]
val_files = list(raw_val.data)[:1]

print("Training cases:", len(train_files))
print("Validation cases:", len(val_files))

In [ ]:
PATCH_SIZE = (64, 64, 32)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-57,
        a_max=164,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=PATCH_SIZE,
        pos=1,
        neg=1,
        num_samples=2,
        image_key="image",
        image_threshold=0,
    ),
    RandFlipd(
        keys=["image", "label"],
        prob=0.5,
        spatial_axis=0,
    ),
    EnsureTyped(keys=["image", "label"]),
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-57,
        a_max=164,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
    EnsureTyped(keys=["image", "label"]),
])

In [ ]:
train_ds = CacheDataset(
    data=train_files,
    transform=train_transforms,
    cache_rate=1.0,
    num_workers=2,
)

val_ds = Dataset(
    data=val_files,
    transform=val_transforms,
)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
)

batch = next(iter(train_loader))

print("Training image batch:", tuple(batch["image"].shape))
print("Training label batch:", tuple(batch["label"].shape))
print("Unique label values:", torch.unique(batch["label"]).tolist())

## Stop and verify

A typical training batch should look like:

```text
[2, 1, 64, 64, 32]
```

Interpret this as:

```text
2 sampled patches
1 channel
64 × 64 × 32 voxels
```

Before moving on, ask:

- Are image and label shapes identical?
- Are the labels still discrete?
- Did the pipeline produce a channel dimension?
- Did the LLM accidentally apply random augmentation to validation?
- Did it use a continuous interpolation method for the segmentation?

> **LLMs can generate code quickly. Good practice is to have it generate code you can verify.**

Most current LLMs are really advanced, and solve **common** problems quite successfully. This is particularly true for a typical **training** exercise such as the spleen segmentation with MONAI. There are **plenty** of examples the LLM could have been trained on, including MONAIs own official tutorial notebooks.

The challenge arises when you want it to do something **out of the ordinary**. Especially in particle therapy, or in new fields of radiotherapy, the LLM will make assumptions based on its training data that may or may not translate well to the sepecific task.

> **Do not trust the pipeline, without veifying that it makes sense for your task!**

# 3. Ask for the model — and nothing else

When you have verified the data pieline works and is reasonable, you can move on to generate a small model architecture for your task. A possible prompt could be:

> The data pipeline works.  
>
> Define only a small 3D MONAI U-Net for binary segmentation.  
>
> Requirements:
> - one CT input channel;
> - one foreground output channel;
> - deliberately small channel counts for free Colab;
> - no pretrained weights;
> - no wrapper class unless MONAI requires it.
>
> Print the number of trainable parameters and show one forward pass on the batch to verify the output shape.  
>
> Do not write the loss or training loop yet.

This is a useful test of whether the LLM respects scope.

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(8, 16, 32),
    strides=(2, 2),
    num_res_units=1,
).to(device)

n_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {n_parameters:,}")

In [ ]:
images = batch["image"].to(device)

with torch.no_grad():
    logits = model(images)

print("Input shape: ", tuple(images.shape))
print("Output shape:", tuple(logits.shape))

## Verification question

For binary segmentation, we expect:

```text
input:
[batch, 1, X, Y, Z]

output:
[batch, 1, X, Y, Z]
```

If the output is:

```text
[batch, 2]
```

the code may run, but the model is solving a completely different problem.

> **Execution is evidence of syntactic compatibility, not scientific correctness.**

The LLM has access to many advanced techniques you may not be familiar with. Since in its core, the LLM is still providing the most likely response for your prompt (and possibly self-gathered context e.g. from web searches), model choices are not expected to be necessarily ideal for your task.

###What to do if I don't know anything about AI?

You can ask the LLM to explain to you in detail what the code does and what pros and cons of the choices made could be. We will come back later to asking the LLM itself to critique and reduce its code. However, you should also read into the tools yourself, checking out the official websites and documentation for the code the AI has generated, particularly if its critical to the project.

> **Using an LLM does not mean you do not need any knowledge of what you are doing**

That should be obvious, but if you are building something, and you do not understand how the code works you built, is this scientific good practice?

Probably not.

I personally would not feel comfortable publishing something of which I myself do not know how it works. You do not need to be able to create the code from scratch, but you should be able to explain it.

# 4. Ask for the training step

Now ask:

> Add only:
> - Dice loss with sigmoid handling;
> - Adam optimizer;
> - a minimal PyTorch training loop for 3 epochs.
>
> Keep the loop explicit. I want to see:
>
> ```python
> optimizer.zero_grad()
> logits = model(images)
> loss = ...
> loss.backward()
> optimizer.step()
> ```
>
> Do not create a Trainer class.
> Do not add schedulers, callbacks, checkpoint managers, or logging frameworks.
> Print the mean training loss after each epoch.
> Keep the mean training loss after each epoch in a list.

This prompt deliberately asks the LLM **not to hide** the learning mechanism. This is particularly useful for wrappers such as MONAI that provide their own Trainer classes. The abstraction hides a lot of the inner working, making it harder for a beginner to verify the settings.

In [ ]:
loss_function = DiceLoss(sigmoid=True)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

N_EPOCHS = 3

train_losses = []

for epoch in range(1, N_EPOCHS + 1):

    model.train()
    running_loss = 0.0
    n_steps = 0

    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].float().to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        n_steps += 1

    epoch_loss = running_loss / n_steps
    train_losses.append(epoch_loss)

    print(
        f"Epoch {epoch:02d}/{N_EPOCHS} | "
        f"training loss: {epoch_loss:.3f}"
    )

# 5. What does the output tell us?

First, the training should be executed without problems. But beyond

> "Did training finish?"

ask:

```text
Did the loss change?
Is the model output the expected shape?
```

A short run is not expected to produce a useful model.

The purpose is to verify the **framework**:

```text
data
→ model
→ loss
→ gradients
→ parameter update
```

If the loss decreases somewhat, that is encouraging.

If it does not even if you increase the number of epochs, that is a debugging problem.

The code in the next cell plots the loss for the provided reference implementation. Since it may not be compatible with your LLMs code, ask it to provide you something similar:

> Provide a plot function that shows the loss against the number of epochs that can be executed after your previous code.

In [1]:
plt.figure(figsize=(6, 4))
plt.plot(
    range(1, N_EPOCHS + 1),
    train_losses,
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Dice loss")
plt.title("Training loss")
plt.show()

NameError: name 'plt' is not defined

# 6. Ask for whole-volume inference

Now ask:

> Training works on patches.  
>
> Add whole-volume validation using MONAI `sliding_window_inference`.  
>
> Requirements:
> - use the same patch size as the sliding-window ROI;
> - apply sigmoid to the logits;
> - threshold at 0.5;
> - compute foreground Dice on the one validation case;
> - do not retrain the model;
> - keep the implementation short.
>
> Explain why sliding-window inference is needed before giving the code.

This is a good example of asking the LLM for both:

```text
implementation
+
reasoning about the implementation
```

In [ ]:
dice_metric = DiceMetric(
    include_background=False,
    reduction="mean",
    num_classes=2,
)

model.eval()
dice_metric.reset()

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        labels = batch["label"].float().to(device)

        logits = sliding_window_inference(
            inputs=images,
            roi_size=PATCH_SIZE,
            sw_batch_size=2,
            predictor=model,
            overlap=0.25,
        )

        probabilities = torch.sigmoid(logits)
        predictions = (probabilities > 0.5).float()

        dice_metric(
            y_pred=predictions,
            y=labels,
        )

val_dice = dice_metric.aggregate().item()

print(f"Validation Dice: {val_dice:.3f}")

## Interpret cautiously

This is:

```text
1 validation patient
+
very few training patients
+
3 epochs
```

The result is **not evidence of model quality**. It is only evidence that the complete pipeline can run.

Ask your LLM to evaluate the model performance. A useful LLM response here should not claim:

> "The model performs well."

based on one patient and a tiny teaching run.

Always challenge confident-sounding language: LLMs tend to be much more confident than real scientists, but may also quickly and completely change their opinion when challenged.

> **Do not view an LLM agreeing with you as confirmation!**

All modern LLMs have been "aligned": Human feedback was used to train them to reflect a personality that most users *like*, which is not necessarily one that challenges their believes.

# 7. Visualize the result

Ask your LLM:

> Add a minimal visualization for the validation case:
> - select the slice containing the largest reference spleen area;
> - show CT;
> - overlay the reference contour;
> - overlay the prediction in a separate panel.
>
> Use matplotlib directly. Do not create visualization helper classes.

Then compare with the reference below.

In [ ]:
val_case = val_ds[0]

image = val_case["image"].unsqueeze(0).to(device)
label = val_case["label"].unsqueeze(0).float().to(device)

model.eval()

with torch.no_grad():
    logits = sliding_window_inference(
        inputs=image,
        roi_size=PATCH_SIZE,
        sw_batch_size=2,
        predictor=model,
        overlap=0.25,
    )

    probability = torch.sigmoid(logits)
    prediction = (probability > 0.5).float()

image_np = image[0, 0].cpu().numpy()
label_np = label[0, 0].cpu().numpy()
pred_np = prediction[0, 0].cpu().numpy()

slice_areas = label_np.sum(axis=(0, 1))
z = int(np.argmax(slice_areas))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(image_np[:, :, z].T, cmap="gray", origin="lower")
axes[0].set_title("CT")

axes[1].imshow(image_np[:, :, z].T, cmap="gray", origin="lower")
axes[1].contour(label_np[:, :, z].T, levels=[0.5])
axes[1].set_title("Reference")

axes[2].imshow(image_np[:, :, z].T, cmap="gray", origin="lower")
if pred_np[:, :, z].max() > 0:
    axes[2].contour(pred_np[:, :, z].T, levels=[0.5])
axes[2].set_title("Prediction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

# 8. Now ask the LLM to reduce the code

You have a complete working pipeline.

Now use the LLM as a reviewer:

> Review the code we built.  
>
> I want to keep the exact same behaviour but make the notebook easier for a beginner to read.
>
> Identify:
> - functions or classes that can be deleted;
> - repeated code that is actually worth simplifying;
> - repeated code that should stay explicit because it helps teaching;
> - unused imports;
> - unnecessary configuration;
> - comments that merely paraphrase Python;
> - library functionality that was reimplemented unnecessarily.
>
> Do not rewrite the notebook yet. Give me a deletion/simplification plan first.

> **After asking an LLM to add code, sometimes ask it to remove code.**

This tends to be important, as LLM code usually are "fluffy": they add a lot of functionality that you may not need, abstractions that are hard to read for beginners etc.

This is not only the case in bigger projects, but particularly also in small, seemingly simple scripts that the LLM tends to overcomplicate. Usually, if you are not specific, it can try to add pointless execution options or safe guards for cases that are never encountered.

As an example, if you ask an LLM for a short script to plot the time progression from two different log files together, without specifying that the logs run on synchronized clocks, it may automatically add code for the case they need to be aligned. As a user, you then could get erroneous behvior, as the code tries to align the starting time of the log files, completely messing with the contained information.


# 9. Compare models

If there are several learners doing this course at the same time and are using different open models, now is a good time to compare results.

If there are no other learners, ask another LLM yourself, if you can access it in an approved way.

Ask the same narrowly specified question:

> Define a minimal 3D binary-segmentation model for this notebook using MONAI.

You may get:

```text
different U-Net sizes
different activations
different normalization
different imports
different explanations
different levels of unnecessary abstraction
```

This is expected.

An LLM prompt is not a deterministic computer program.

> **The implementation provided by a particular LLM is not necessarily uniquely correct nor even good.**

# 10. What have we actually achieved?

With the help of an LLM, we have built:

```text
public CT dataset
→ preprocessing
→ training patches
→ 3D neural network
→ loss
→ optimization
→ sliding-window inference
→ Dice evaluation
→ visual prediction
```

That is a real, if deliberately tiny, medical-image AI workflow.

But the important lesson is not:

> "The LLM knows how to build a U-Net."

LLMs only ever provide the most likely response to your input. Hence, the important lesson to take away here is:

> **How to guide the LLM through a sequence of bounded tasks and verify each result.**

Reflect on what you have learned so far, and be sure that you understood how to prompt in future tasks.

# 11. Exit exercise — what would you improve next?

Let's try to improve our model. However, do not ask the LLM something generic like:

> "Make the model better."

Instead, choose one concrete hypothesis.

Examples:

```text
Train for more epochs.
Use more training cases.
Change the patch size.
Add one plausible augmentation.
Use a larger model.
Inspect false-positive regions.
Compare Dice before and after one change.
```

Then write a prompt that contains:

```text
current state
one proposed change
reason for the change
what should remain unchanged
how success will be measured
```

Example:

> The current model trains on four cases for three epochs and achieves a validation Dice of X on one case.  
>
> I want to test only whether training longer improves the result.  
>
> Change the notebook to train for ten epochs. Do not change the model, data, transforms, optimizer, or validation procedure. Keep the existing loss plot and validation Dice calculation.


# Take-home message

A useful LLM coding workflow looks like:

```text
specify
↓
plan
↓
implement one step
↓
run
↓
verify
↓
simplify
↓
change one thing
↓
measure again
```

The LLM helps generate possibilities.

The computer provides evidence.

You decide what the evidence means.

---

## Bridge to Lesson 4

Our code now works.

That does not mean it is good code.

In the next lesson we will deliberately examine common problems in LLM-generated programs:

- unnecessary helper functions;
- abstraction for its own sake;
- duplicated functionality;
- verbose comments;
- silent scientific errors;
- "fixes" that remove useful checks;
- code that is longer but not more capable.

> **How do we debug not only whether LLM code runs, but whether it is actually necessary?**